In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel, cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer
from ast import literal_eval
import numpy as np

In [ ]:
# Cargar dataset principal

df = pd.read_csv('./data/movies_metadata.csv', low_memory=False)

df.head()

In [ ]:
# Recomendador simple (top películas)

C = df['vote_average'].mean()

print(C)

In [ ]:
# Calcular mínimo de votos (percentil 90)

m = df['vote_count'].quantile(0.90)

print(m)

In [ ]:
# Filtrar películas que cumplan con suficientes votos

q_movies = df.copy().loc[df['vote_count'] >= m]

q_movies.shape

In [ ]:
# Función de media ponderada

def weighted_rating(x, m=m, C=C):
    
    v = x['vote_count']
    R = x['vote_average']
    
    return (v/(v+m) * R) + (m/(m+v) * C)

In [ ]:
# Calcular score

q_movies['score'] = q_movies.apply(weighted_rating, axis=1)

q_movies = q_movies.sort_values('score', ascending=False)

q_movies[['title','vote_count','vote_average','score']].head(15)

In [ ]:
## Recomendador basado en contenido (sinopsis)

df['overview'] = df['overview'].fillna('')

In [ ]:
# Vectorización TF-IDF

tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(df['overview'])

tfidf_matrix.shape

In [ ]:
# Calcular similitud coseno

cosine_similarity(tfidf_matrix, tfidf_matrix)

cosine_sim.shape

In [ ]:
# Crear mapa de índices

indices = pd.Series(df.index, index=df['title']).drop_duplicates()

indices[:10]

In [ ]:
# Función recomendadora

def get_recommendations(title, cosine_sim=cosine_sim):
    
    idx = indices[title]
    
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    sim_scores = sim_scores[1:11]
    
    movie_indices = [i[0] for i in sim_scores]
    
    return df['title'].iloc[movie_indices]

In [ ]:
# Probar recomendador
get_recommendations('The Dark Knight Rises')

In [ ]:
get_recommendations('The Godfather')

In [ ]:
## Recomendador mejorador con metadatos
# Cargar datasets adicionales

credits = pd.read_csv('./data/credits.csv')
keywords = pd.read_csv('./data/keywords.csv')

In [ ]:
# Limpiar IDs
df = df.drop([19730, 29503, 35587, 35803])

In [ ]:
# Merge de datasets
df = df.merge(credits, on='id')
df = df.merge(keywords, on='id')

In [ ]:
# Convertir strings en listas
features = ['cast', 'crew', 'keywords', 'genres']

for feature in features:
    df[feature] = df[feature].apply(literal_eval)

In [ ]:
# Función para obtener difector
def get_director(x):
    for i in x:
        if i['job'] == 'Director':
            return i['name']
    return np.nan

In [ ]:
# Función para obtener top 3
def get_list(x):
    if isinstance(x, list):
        names = [i['name'] for i in x]
        if len(names) > 3:
            names = names[:3]
        return names
    return []

In [ ]:
# Aplicar funciones
df['director'] = df['crew'].apply(get_director)
df['crew'] = df['cast'].apply(get_list)

features = ['cast', 'keywords', 'genres']

for feature in features:
    df[feature] = df[feature].apply(get_list)

In [ ]:
# Limpieza de texto

def clean_data(x):

    if isinstance(x, list):
        return [str.lower(i.replace(" ", "")) for i in x]
    else:
        if isinstance(x, str):
            return str.lower(x.replace(" ", ""))
        else:
            return ''

In [ ]:
# Aplicar limpieza
features = ['cast', 'crew', 'keywords', 'genres']

for feature in features:
    df[feature] = df[feature].apply(clean_data)

In [ ]:
# Creaar metadata sopu

def create_soup(x):
    return ' '.join(x['keywords']) + ' ' + ' '.join(x['cast']) + ' ' + x['director'] + ' ' + ' '.join(x['genres'])


In [ ]:
df['soup'] = df.apply(create_soup, axis=1)

df[['title', 'soup']].head()

In [ ]:
# Vectorización CountVectorizer

count = CountVectorizer(stop_words='english')

count_matrix = count.fit_transform(df['soup'])

count_matrix.shape

In [ ]:
# Similitud coseno

cosine_sim2 = cosine_similarity(count_matrix, count_matrix)

In [ ]:
# Reindexar
df = df.reset_index()
indices = pd.Series(df.index, index=df['title'])


In [ ]:
# Probar recomendador mejorado

get_recommendations('The Dark Knight Rises', cosine_sim2)

In [ ]:
get_recommendations('The godfather', cosine_sim2)

In [ ]:
# Sistema interactivo

movie = input('Ingrese el título de una película: ')
get_recommendations(movie, cosine_sim2)
